In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime
from functools import reduce
 
CONFIG = {
    "catalog": "workspace",
    "schema":  "banking_datasentry",
}
 
SCORED_AT = datetime.now().isoformat()
 
# Health status thresholds
THRESHOLDS = {
    "GREEN": 99.0,   # >= 99% pass rate
    "AMBER": 95.0,   # >= 95% pass rate
    "RED":   0.0,    # < 95% pass rate
}
 
spark.sql(f"USE CATALOG {CONFIG['catalog']}")
spark.sql(f"USE SCHEMA {CONFIG['schema']}")
 
print(f"Catalog    : {CONFIG['catalog']}")
print(f"Schema     : {CONFIG['schema']}")
print(f"Scored at  : {SCORED_AT}")
print(f"Thresholds : GREEN >= {THRESHOLDS['GREEN']}% | AMBER >= {THRESHOLDS['AMBER']}% | RED < {THRESHOLDS['AMBER']}%")

Catalog    : workspace
Schema     : banking_datasentry
Scored at  : 2026-04-25T21:14:09.301602
Thresholds : GREEN >= 99.0% | AMBER >= 95.0% | RED < 95.0%


In [0]:
#Assigning Health Status
def assign_health_status(pass_rate):
    if pass_rate is None:
        return "RED"
    if pass_rate >= THRESHOLDS["GREEN"]:
        return "GREEN"
    elif pass_rate >= THRESHOLDS["AMBER"]:
        return "AMBER"
    else:
        return "RED"
 
health_status_udf = F.udf(assign_health_status, T.StringType())
 
print("health_status_udf registered.")

health_status_udf registered.


In [0]:
#Extracting Rule Scores from a Silver Table
def extract_rule_scores(silver_df, table_name: str, rule_dimension_map: dict):
    """
    Compute pass/fail counts and pass rate for each rule column in silver_df.
    Returns a DataFrame with one row per rule.
    """
    total = silver_df.count()
    rule_cols = [c for c in silver_df.columns if c.startswith("rule_")]
    rows = []
 
    for col in rule_cols:
        rule_name = col.replace("rule_", "")
        pass_count = silver_df.filter(F.col(col) == 1).count()
        fail_count = total - pass_count
        pass_rate  = round((pass_count / total) * 100, 2) if total > 0 else 0.0
        dimension  = rule_dimension_map.get(rule_name, "Unknown")
 
        rows.append((
            table_name,
            rule_name,
            dimension,
            total,
            pass_count,
            fail_count,
            pass_rate,
            SCORED_AT,
        ))
 
    schema = T.StructType([
        T.StructField("table_name",     T.StringType(),  False),
        T.StructField("rule_name",      T.StringType(),  False),
        T.StructField("dimension",      T.StringType(),  False),
        T.StructField("total_records",  T.LongType(),    False),
        T.StructField("pass_count",     T.LongType(),    False),
        T.StructField("fail_count",     T.LongType(),    False),
        T.StructField("pass_rate",      T.DoubleType(),  False),
        T.StructField("scored_at",      T.StringType(),  False),
    ])
 
    return spark.createDataFrame(rows, schema)

In [0]:
#Rule-to-Dimension Maps
customer_rule_dimension_map = {
    "customer_id_not_null":       "Completeness",
    "full_name_not_null":         "Completeness",
    "email_not_null":             "Completeness",
    "created_at_not_null":        "Completeness",
    "email_format_valid":         "Validity",
    "date_of_birth_valid":        "Validity",
    "country_code_valid":         "Validity",
    "customer_id_uuid_format":    "Validity",
    "created_at_not_future":      "Consistency",
    "date_of_birth_not_future":   "Consistency",
    "age_between_18_and_120":     "Consistency",
    "customer_id_unique":         "Uniqueness",
}

account_rule_dimension_map = {
    "account_id_not_null":                  "Completeness",
    "customer_id_not_null":                 "Completeness",
    "account_type_not_null":                "Completeness",
    "status_not_null":                      "Completeness",
    "account_type_valid":                   "Validity",
    "status_valid":                         "Validity",
    "currency_valid":                       "Validity",
    "account_id_uuid_format":               "Validity",
    "opened_date_valid":                    "Validity",
    "balance_valid":                        "Validity",      # NEW
    "savings_loan_balance_not_negative":    "Consistency",
    "opened_date_not_future":               "Consistency",
    "account_id_unique":                    "Uniqueness",
    "customer_id_exists_in_customers":      "Referential Integrity",
}

transaction_rule_dimension_map = {
    "transaction_id_not_null":              "Completeness",
    "account_id_not_null":                  "Completeness",
    "transaction_type_not_null":            "Completeness",
    "amount_not_null":                      "Completeness",
    "currency_not_null":                    "Completeness",
    "transaction_date_not_null":            "Completeness",
    "status_not_null":                      "Completeness",
    "transaction_type_valid":               "Validity",
    "status_valid":                         "Validity",
    "currency_valid":                       "Validity",
    "transaction_id_uuid_format":           "Validity",
    "amount_greater_than_zero":             "Validity",
    "amount_valid":                         "Validity",      # NEW
    "transaction_date_not_future":          "Consistency",
    "failed_reversed_amount_not_positive":  "Consistency",
    "transaction_id_unique":                "Uniqueness",
    "account_id_exists_in_accounts":        "Referential Integrity",
}

print("Rule-to-dimension maps defined.")
print(f"  Customers    : {len(customer_rule_dimension_map)} rules")
print(f"  Accounts     : {len(account_rule_dimension_map)} rules")
print(f"  Transactions : {len(transaction_rule_dimension_map)} rules")

Rule-to-dimension maps defined.
  Customers    : 12 rules
  Accounts     : 13 rules
  Transactions : 16 rules


In [0]:
silver_customers    = spark.read.format("delta").table("silver_customers")
silver_accounts     = spark.read.format("delta").table("silver_accounts")
silver_transactions = spark.read.format("delta").table("silver_transactions")

print("Silver tables loaded:")
print(f"  silver_customers     : {silver_customers.count():,} rows")
print(f"  silver_accounts      : {silver_accounts.count():,} rows")
print(f"  silver_transactions  : {silver_transactions.count():,} rows")

Silver tables loaded:
  silver_customers     : 50,500 rows
  silver_accounts      : 55,000 rows
  silver_transactions  : 202,000 rows


In [0]:
#Computing Rule Level Scores
customer_rule_scores    = extract_rule_scores(silver_customers,    "customers",    customer_rule_dimension_map)
account_rule_scores     = extract_rule_scores(silver_accounts,     "accounts",     account_rule_dimension_map)
transaction_rule_scores = extract_rule_scores(silver_transactions,  "transactions", transaction_rule_dimension_map)
 
# Union all three into one scorecard
gold_rule_scores = customer_rule_scores \
    .union(account_rule_scores) \
    .union(transaction_rule_scores)
 
# Add health status column
gold_rule_scores = gold_rule_scores.withColumn(
    "health_status",
    health_status_udf(F.col("pass_rate"))
)
 
# Write to Gold Delta table
gold_rule_scores.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_rule_scores")
 
print(f"gold_rule_scores written: {gold_rule_scores.count()} rows")
gold_rule_scores.orderBy("table_name", "dimension", "pass_rate").show(50, truncate=50)

gold_rule_scores written: 41 rows
+------------+-----------------------------------+---------------------+-------------+----------+----------+---------+--------------------------+-------------+
|  table_name|                          rule_name|            dimension|total_records|pass_count|fail_count|pass_rate|                 scored_at|health_status|
+------------+-----------------------------------+---------------------+-------------+----------+----------+---------+--------------------------+-------------+
|    accounts|                account_id_not_null|         Completeness|        55000|     55000|         0|    100.0|2026-04-25T21:14:09.301602|        GREEN|
|    accounts|               customer_id_not_null|         Completeness|        55000|     55000|         0|    100.0|2026-04-25T21:14:09.301602|        GREEN|
|    accounts|              account_type_not_null|         Completeness|        55000|     55000|         0|    100.0|2026-04-25T21:14:09.301602|        GREEN|
|    a

In [0]:
#Computing Dimension Level Scores
gold_dimension_scores = gold_rule_scores \
    .groupBy("table_name", "dimension") \
    .agg(
        F.sum("total_records").alias("total_checks"),
        F.sum("pass_count").alias("total_passes"),
        F.sum("fail_count").alias("total_fails"),
        F.count("rule_name").alias("rule_count"),
    ) \
    .withColumn(
        "pass_rate",
        F.round(F.col("total_passes") / F.col("total_checks") * 100, 2)
    ) \
    .withColumn("scored_at", F.lit(SCORED_AT))
 
# Add health status
gold_dimension_scores = gold_dimension_scores.withColumn(
    "health_status",
    health_status_udf(F.col("pass_rate"))
)
 
# Write to Gold Delta table
gold_dimension_scores.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dimension_scores")
 
print(f"gold_dimension_scores written: {gold_dimension_scores.count()} rows")
gold_dimension_scores.orderBy("table_name", "pass_rate").show(30, truncate=50)

gold_dimension_scores written: 14 rows
+------------+---------------------+------------+------------+-----------+----------+---------+--------------------------+-------------+
|  table_name|            dimension|total_checks|total_passes|total_fails|rule_count|pass_rate|                 scored_at|health_status|
+------------+---------------------+------------+------------+-----------+----------+---------+--------------------------+-------------+
|    accounts|Referential Integrity|       55000|       52280|       2720|         1|    95.05|2026-04-25T21:14:09.301602|        AMBER|
|    accounts|             Validity|      275000|      275000|          0|         5|    100.0|2026-04-25T21:14:09.301602|        GREEN|
|    accounts|         Completeness|      220000|      220000|          0|         4|    100.0|2026-04-25T21:14:09.301602|        GREEN|
|    accounts|          Consistency|      110000|      110000|          0|         2|    100.0|2026-04-25T21:14:09.301602|        GREEN|
| 

In [0]:
#Computing Table Level Scores
gold_table_scores = gold_rule_scores \
    .groupBy("table_name") \
    .agg(
        F.sum("total_records").alias("total_checks"),
        F.sum("pass_count").alias("total_passes"),
        F.sum("fail_count").alias("total_fails"),
        F.count("rule_name").alias("total_rules"),
    ) \
    .withColumn(
        "pass_rate",
        F.round(F.col("total_passes") / F.col("total_checks") * 100, 2)
    ) \
    .withColumn("scored_at", F.lit(SCORED_AT))
 
# Add health status per table
gold_table_scores = gold_table_scores.withColumn(
    "health_status",
    health_status_udf(F.col("pass_rate"))
)
 
# Write to Gold Delta table
gold_table_scores.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_table_scores")
 
print(f"gold_table_scores written: {gold_table_scores.count()} rows")
gold_table_scores.orderBy("table_name").show(truncate=50)

gold_table_scores written: 3 rows
+------------+------------+------------+-----------+-----------+---------+--------------------------+-------------+
|  table_name|total_checks|total_passes|total_fails|total_rules|pass_rate|                 scored_at|health_status|
+------------+------------+------------+-----------+-----------+---------+--------------------------+-------------+
|    accounts|      715000|      712280|       2720|         13|    99.62|2026-04-25T21:14:09.301602|        GREEN|
|   customers|      606000|      603993|       2007|         12|    99.67|2026-04-25T21:14:09.301602|        GREEN|
|transactions|     3232000|     3186820|      45180|         16|     98.6|2026-04-25T21:14:09.301602|        AMBER|
+------------+------------+------------+-----------+-----------+---------+--------------------------+-------------+



In [0]:
#Pipeline Health Summary
print("=" * 60)
print("DATASENTRY — PIPELINE HEALTH SUMMARY")
print("=" * 60)
 
table_scores = gold_table_scores.collect()
total_checks  = sum(r["total_checks"]  for r in table_scores)
total_passes  = sum(r["total_passes"]  for r in table_scores)
total_fails   = sum(r["total_fails"]   for r in table_scores)
overall_rate  = round(total_passes / total_checks * 100, 2)
 
print(f"\nOverall Pipeline Pass Rate : {overall_rate}%")
print(f"Total Checks               : {total_checks:,}")
print(f"Total Passes               : {total_passes:,}")
print(f"Total Fails                : {total_fails:,}")
 
print(f"\n{'TABLE':<20} {'RULES':>6} {'PASS RATE':>10} {'STATUS':>8}")
print("-" * 48)
for r in sorted(table_scores, key=lambda x: x["table_name"]):
    print(f"{r['table_name']:<20} {r['total_rules']:>6} {r['pass_rate']:>9}% {r['health_status']:>8}")
 
print("\nNotebook 04 complete. Gold scoring tables ready for Power BI.")

DATASENTRY — PIPELINE HEALTH SUMMARY

Overall Pipeline Pass Rate : 98.9%
Total Checks               : 4,553,000
Total Passes               : 4,503,093
Total Fails                : 49,907

TABLE                 RULES  PASS RATE   STATUS
------------------------------------------------
accounts                 13     99.62%    GREEN
customers                12     99.67%    GREEN
transactions             16      98.6%    AMBER

Notebook 04 complete. Gold scoring tables ready for Power BI.


In [0]:
#Gold: Top 10 Failing Rules (pre-aggregated for Power BI)
# Pre-aggregates the top 10 failing rules so Power BI does not need
# to filter — the Gold layer serves ready-to-consume data.

gold_top10_rules = gold_rule_scores \
    .orderBy(F.desc("fail_count")) \
    .limit(10)

gold_top10_rules.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_top10_failing_rules")

print("gold_top10_failing_rules written:")
gold_top10_rules \
    .select("table_name", "rule_name", "fail_count", "pass_rate", "health_status") \
    .show(truncate=50)